In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import sys
sys.path.append('./')
sys.path.append('/Users/ASUS/Documents/GitHub/retrospective_t2e_analysis/auton-survival/')

from gls import datasets

import pandas as pd
import numpy as np

In [21]:
def load_sprint_trial_data(outcome=''):
  """Helper function to load and preprocess the SPRINT dataset.

  The SPRINT Dataset is a subset of 9,361 participants of the well known
  Systolic Blood Pressure Intervention Trial [1] for studying systolic 
  blood pressure to reduce cardiovascular morbidity and mortality among persons 
  without diabetes. It is a popular dataset for longitudinal survival analysis 
  with time dependent covariates.

  References
  ----------
  [1]“A Randomized Trial of Intensive versus Standard Blood-Pressure Control,” 
  N Engl J Med, vol. 373, no. 22, pp. 2103–2116, Nov. 2015, doi: 10.1056/NEJMoa1511939.

  Website and Documentation
  ----------
  [1] https://biolincc.nhlbi.nih.gov/studies/sprint/
  [2] https://biolincc.nhlbi.nih.gov/media/studies/sprint/data_dictionary/SPRINT_2020b.pdf?link_time=2022-07-08_11:48:02.911607
  """

  feature_list = {
    # demographic data
    'baseline.sas7bdat' : ['INTENSIVE', 'RISK10YRS', 'INCLUSIONFRS', 'SBP', 'DBP', 'N_AGENTS', 'NOAGENTS', 
                           'SMOKE_3CAT', 'ASPIRIN', 'EGFR', 'SCREAT', 'SUB_CKD', 'RACE_BLACK', 'AGE', 'FEMALE', 
                           'SUB_CVD', 'SUB_CLINICALCVD', 'SUB_SUBCLINICALCVD', 'SUB_SENIOR', 'RACE4', 'CHR', 'GLUR', 
                           'HDL', 'TRR', 'UMALCR', 'BMI', 'STATIN', 'SBPTERTILE']
  }

  outcomedt = outcome.replace('EVENT', 'T')

  outcomes, features = datasets._load_generic_biolincc_dataset(outcome_tbl='outcomes.sas7bdat', 
                                                      time_col= outcomedt, 
                                                      event_col= outcome,
                                                      features=feature_list, 
                                                      id_col='MASKID',
                                                      location='datasets/SPRINT_POP/Datasets/')

  # Convert Censoring Indicator to Binary
  # ie. 1 = Event, 0 = Censored
  outcomes.event = outcomes.event == 1.0 
  
  NaNindex = pd.isna(outcomes.time)
  outcomes = outcomes.loc[~NaNindex]
  features = features.loc[~NaNindex]

  features = features.rename(columns={"INTENSIVE": "Intervention", 
                                      "RISK10YRS": "Estimation_10-year_Casdiovascular_disease_risk",
                                      "INCLUSIONFRS": "10-year_Casdiovascular_disease_risk>15%",
                                      "SBP": "Seated_Systolic_Blood_Pressure",
                                      "DBP": "Seated_Diastolic_Blood_Pressure",
                                      "N_AGENTS": "Number_medications_prescribed",
                                      "NOAGENTS": "On_no_anti-hypertensive_agents",
                                      "SMOKE_3CAT": "Baseline_Smoke_status",
                                      "ASPIRIN": "Baseline_History_Aspirin",
                                      "EGFR": "Estimated_Glomerular_Filtration_Rate",
                                      "SCREAT": "Serum_creatinine",
                                      "SUB_CKD": "History_Chronic_Kidney_Disease",
                                      "RACE_BLACK": "Is_Black/African-American",
                                      "AGE": "Age", 
                                      "FEMALE": "Sex",
                                      "SUB_CVD": "History_Cardiovascular_Disease",
                                      "SUB_CLINICALCVD": "History_Clinical_Cardiovascular_Disease", 
                                      "SUB_SUBCLINICALCVD": "History_Subclinical_Cardiovascular_Disease", 
                                      "SUB_SENIOR": ">75_years_old", 
                                      "RACE4": "Race", 
                                      "CHR": "Cholesterol", 
                                      "GLUR": "Glucose", 
                                      "HDL": "HDL_Cholesterol",
                                      "TRR": "Triglycerides", 
                                      "UMALCR": "Urine_albumin", 
                                      "STATIN": "On_Statin", 
                                      "SBPTERTILE": "Systolic_Blood_Pressure_Tertile"})

  features['Intervention'].replace({0.0: "SBP target <140 mm Hg", 1.0: "SBP target <120 mm Hg"}, inplace=True)
  features['Baseline_Smoke_status'].replace({1.0: "Never", 2.0: "Former", 3.0: "Current", 4.0: "Missing"}, inplace=True)
  features['On_no_anti-hypertensive_agents'].replace({0.0: "One or more", 1.0: "On no agents"}, inplace=True)
  features['Sex'].replace({0.0: "Male", 1.0: "Female"}, inplace=True)

  cols = ["10-year_Casdiovascular_disease_risk>15%",
          "Baseline_History_Aspirin",
          "History_Chronic_Kidney_Disease",
          "Is_Black/African-American",
          "History_Cardiovascular_Disease",
          "History_Clinical_Cardiovascular_Disease",
          "History_Subclinical_Cardiovascular_Disease",
          ">75_years_old",
          "On_Statin"]
  for col in cols:
    features[col].replace({1.0: "Yes", 0.0: "No"}, inplace=True)

  return outcomes, features

In [22]:
# Load the sprint dataset
outcomes, x = load_sprint_trial_data(outcome='EVENT_PRIMARY')

# Let's take a look at the dataset
x.head()

(9361, 28)


c:\Users\ASUS\Documents\GitHub\retrospective_t2e_analysis\auton-survival\auton_survival\gls\datasets.py:12: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df.loc[:, col] = df[col].values.astype(str)
C:\Users\ASUS\AppData\Local\Temp\ipykernel_33764\568786531.py:40: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  outcomes.event = outcomes.event == 1.0


,On_no_anti-hypertensive_agents,History_Subclinical_Cardiovascular_Disease,Age,Race,Cholesterol,Intervention,History_Chronic_Kidney_Disease,BMI,Is_Black/African-American,Systolic_Blood_Pressure_Tertile,...,Seated_Systolic_Blood_Pressure,History_Clinical_Cardiovascular_Disease,Baseline_Smoke_status,Seated_Diastolic_Blood_Pressure,On_Statin,10-year_Casdiovascular_disease_risk>15%,Baseline_History_Aspirin,Triglycerides,Estimated_Glomerular_Filtration_Rate,Number_medications_prescribed
S00007,One or more,No,60.0,WHITE,155.0,SBP target <140 mm Hg,No,33.115201,No,3.0,...,145.0,Yes,Current,80.0,Yes,Yes,Yes,92.0,67.69,2.0
S00010,One or more,No,75.0,WHITE,243.0,SBP target <140 mm Hg,No,28.842380,No,2.0,...,138.0,No,Former,71.0,Yes,Yes,Yes,188.0,60.64,1.0
S00022,One or more,No,56.0,HISPANIC,232.0,SBP target <120 mm Hg,No,24.888358,No,2.0,...,144.0,No,Never,84.0,Yes,No,No,183.0,85.09,3.0
S00038,One or more,No,62.0,WHITE,180.0,SBP target <120 mm Hg,No,33.643060,No,2.0,...,143.0,No,Former,92.0,No,Yes,No,125.0,68.44,2.0
S00045,One or more,No,75.0,WHITE,234.0,SBP target <140 mm Hg,No,29.337871,No,1.0,...,123.0,No,Never,68.0,No,No,No,109.0,71.94,2.0
